## Cell 1 – Imports and dataset paths

This cell sets up all required libraries and defines the base paths for the project.  
It also creates the `labels_yolo` directory if it does not exist and specifies the target class mapping (`helmet → 0`) that will be used throughout preprocessing.

In [ ]:
import os
import xml.etree.ElementTree as ET

import cv2
import numpy as np

import random
import csv

BASE_DIR = r"C:\Users\ataka\Desktop\Avanced_Math\final_project"
IMAGES_DIR = os.path.join(BASE_DIR, "images")
XML_DIR = os.path.join(BASE_DIR, "annotations")
LABELS_DIR = os.path.join(BASE_DIR, "labels_yolo")
SPLIT_CSV_PATH = os.path.join(BASE_DIR, "splits.csv")

os.makedirs(LABELS_DIR, exist_ok=True)

TARGET_CODES = {
    "helmet": 0,
    # add more if you see them, for example:
    # "person": 1,
    # "head": 2,
}

CLASS_NAMES = {
    0: "helmet",
    # later, if you add more classes:
    # 1: "person",
    # 2: "head",
}

## Cell 2 – Convert XML annotations to YOLO labels

This cell defines functions that:

- Parse PASCAL VOC XML files to read image size and `helmet` objects.
- Convert each bounding box to YOLO-style `(class_id, x_center_norm, y_center_norm, width_norm, height_norm)` coordinates.
- Save one `.txt` label file per image under `labels_yolo`.

In [ ]:
def convert_xml_to_yolo(xml_path: str):
    """Converts a single XML file to YOLO-style labels."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    size = root.find("size")
    img_w = int(size.find("width").text)
    img_h = int(size.find("height").text)

    yolo_labels = []

    for obj in root.findall("object"):
        code = obj.find("name").text.strip()

        if code not in TARGET_CODES:
            continue

        class_id = TARGET_CODES[code]

        bndbox = obj.find("bndbox")
        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        box_w = xmax - xmin
        box_h = ymax - ymin
        x_center = xmin + box_w / 2.0
        y_center = ymin + box_h / 2.0

        x_center_norm = x_center / img_w
        y_center_norm = y_center / img_h
        box_w_norm = box_w / img_w
        box_h_norm = box_h / img_h

        yolo_labels.append(
            (class_id, x_center_norm, y_center_norm, box_w_norm, box_h_norm)
        )

    return yolo_labels


def save_yolo_labels_for_xml(xml_path: str, labels_dir: str) -> None:
    """Saves YOLO-style labels for a single XML file."""
    basename = os.path.splitext(os.path.basename(xml_path))[0]
    label_path = os.path.join(labels_dir, basename + ".txt")

    yolo_labels = convert_xml_to_yolo(xml_path)

    with open(label_path, "w", encoding="utf-8") as f:
        for class_id, xc, yc, w, h in yolo_labels:
            line = f"{class_id} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n"
            f.write(line)


def convert_all_xml_in_dir(xml_dir: str, labels_dir: str) -> None:
    """Converts all XML files in a directory to YOLO label files."""
    xml_files = sorted(
        f for f in os.listdir(xml_dir) if f.lower().endswith(".xml")
    )

    for xml_name in xml_files:
        xml_path = os.path.join(xml_dir, xml_name)
        save_yolo_labels_for_xml(xml_path, labels_dir)

## Cell 3 – Visualize YOLO labels with OpenCV

This cell defines a helper function to draw bounding boxes and class names on an image  
using the generated YOLO `.txt` label file. It is used to visually verify that the  
XML-to-YOLO conversion and normalized coordinates are correct.

In [ ]:
def visualize_yolo_boxes(image_path: str, label_path: str, class_names=None) -> None:
    """Loads an image and its YOLO label file, draws boxes, and shows the result."""
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not load image: {image_path}")
        return

    h, w = image.shape[:2]

    if not os.path.exists(label_path):
        print(f"Label file not found: {label_path}")
        return

    with open(label_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        parts = line.strip().split()
        if len(parts) != 5:
            continue

        class_id = int(parts[0])
        xc = float(parts[1]) * w
        yc = float(parts[2]) * h
        bw = float(parts[3]) * w
        bh = float(parts[4]) * h

        xmin = int(xc - bw / 2.0)
        ymin = int(yc - bh / 2.0)
        xmax = int(xc + bw / 2.0)
        ymax = int(yc + bh / 2.0)

        color = (0, 255, 0)
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), color, 2)

        if class_names is not None:
            name = class_names.get(class_id, str(class_id))
            cv2.putText(
                image,
                name,
                (xmin, max(ymin - 5, 0)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                1,
                cv2.LINE_AA,
            )

## Cell 4 – Train/val/test split and image normalization

This cell:

- Creates a 70/15/15 train/validation/test split over all images and writes `splits.csv`.
- Implements `load_and_normalize_image` to prepare images (RGB, `[0, 1]` floats) for the model.
- Adds a helper `summarize_splits` to print how many images are in each split.

In [ ]:
def create_train_val_test_split(
    images_dir: str,
    split_csv_path: str,
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
) -> None:
    """Creates a train/val/test split and saves it to a CSV file."""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1."

    image_files = sorted(
        f for f in os.listdir(images_dir)
        if f.lower().endswith(".png") or f.lower().endswith(".jpg")
    )

    random.shuffle(image_files)

    n = len(image_files)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    n_test = n - n_train - n_val

    train_files = image_files[:n_train]
    val_files = image_files[n_train:n_train + n_val]
    test_files = image_files[n_train + n_val:]

    assert len(train_files) == n_train
    assert len(val_files) == n_val
    assert len(test_files) == n_test

    with open(split_csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["image_name", "split"])
        for name in train_files:
            writer.writerow([name, "train"])
        for name in val_files:
            writer.writerow([name, "val"])
        for name in test_files:
            writer.writerow([name, "test"])

    print(
        f"Saved split CSV to {split_csv_path} with {n_train} train, {n_val} val, {n_test} test samples."
    )


def load_and_normalize_image(image_path: str):
    """Loads an image and normalizes it for model input (RGB, [0, 1])."""
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image.astype(np.float32) / 255.0
    return image


def summarize_splits(split_csv_path: str):
    """Counts how many images are in each split (train/val/test)."""
    counts = {"train": 0, "val": 0, "test": 0}
    with open(split_csv_path, "r", encoding="utf-8") as f:
        next(f)
        for line in f:
            _, split = line.strip().split(",")
            counts[split] += 1
    print("Split summary:", counts)

## Cell 5 – Run preprocessing and show sample visualization

This cell runs the end-to-end preprocessing:

- Converts all XML files to YOLO labels.
- Creates the train/validation/test split.
- Prints a split summary.
- Visualizes one sample image with its helmet bounding boxes to confirm correctness.

In [ ]:
convert_all_xml_in_dir(XML_DIR, LABELS_DIR)
create_train_val_test_split(IMAGES_DIR, SPLIT_CSV_PATH)

sample_image = os.path.join(IMAGES_DIR, "hard_hat_workers1.png")
sample_label = os.path.join(LABELS_DIR, "hard_hat_workers1.txt")

print("Image path:", sample_image, "Exists:", os.path.exists(sample_image))
print("Label path:", sample_label, "Exists:", os.path.exists(sample_label))

visualize_yolo_boxes(sample_image, sample_label, class_names=CLASS_NAMES)
summarize_splits(SPLIT_CSV_PATH)